In [2]:
import duckdb
import pandas as pd

con = duckdb.connect("../data/processed/provider_quality.db")

# Parameters — this is what a recommendation engine would receive as input
## Case 1: New York — Heart Attack
state = "NY"
condition = "MORT_30_AMI"  # Heart attack

print(f"Finding top hospitals in {state} for {condition}")

Finding top hospitals in NY for MORT_30_AMI


In [3]:
# Top 5 hospital recommendation for a given state and condition
recommendations = con.execute(f"""
    WITH eligible_hospitals AS (
        SELECT
            c."Facility ID",
            c."Facility Name",
            h."City/Town"        as city,
            h."State",
            h."Hospital Type",
            h."Hospital Ownership",
            c."Measure ID",
            c."Measure Name",
            CAST(c."Score" AS FLOAT)         as score,
            CAST(c."Denominator" AS INTEGER)  as denominator,
            c."Compared to National"
        FROM complications c
        JOIN hospitals h ON c."Facility ID" = h."Facility ID"
        WHERE h."State" = '{state}'
          AND c."Measure ID" = '{condition}'
          AND c."Score" NOT IN ('Not Available', 'Not Applicable')
          AND c."Denominator" NOT IN ('Not Available', 'Not Applicable')
          AND CAST(c."Denominator" AS INTEGER) >= 25
    )
    SELECT
        "Facility Name",
        city,
        "Hospital Type",
        "Hospital Ownership",
        "Measure Name",
        score                                           as mortality_rate,
        denominator                                     as patients_evaluated,
        "Compared to National",
        RANK() OVER (ORDER BY score ASC)                as recommendation_rank
    FROM eligible_hospitals
    ORDER BY recommendation_rank
    LIMIT 5
""").df()

recommendations

,Facility Name,city,Hospital Type,Hospital Ownership,Measure Name,mortality_rate,patients_evaluated,Compared to National,recommendation_rank
0,NYU LANGONE HOSPITALS,NEW YORK,Acute Care Hospitals,Voluntary non-profit - Private,Death rate for heart attack patients,6.7,633,Better Than the National Rate,1
1,MONTEFIORE MEDICAL CENTER,BRONX,Acute Care Hospitals,Voluntary non-profit - Private,Death rate for heart attack patients,9.6,275,Better Than the National Rate,2
2,NORTHERN WESTCHESTER HOSPITAL,MOUNT KISCO,Acute Care Hospitals,Voluntary non-profit - Private,Death rate for heart attack patients,9.9,178,No Different Than the National Rate,3
3,NEW YORK-PRESBYTERIAN HOSPITAL,NEW YORK,Acute Care Hospitals,Voluntary non-profit - Private,Death rate for heart attack patients,9.9,613,Better Than the National Rate,3
4,MAIMONIDES MEDICAL CENTER,BROOKLYN,Acute Care Hospitals,Voluntary non-profit - Private,Death rate for heart attack patients,10.2,399,No Different Than the National Rate,5


In [6]:
# Add gap vs #1 hospital to show non-linear differences
best_score = recommendations['mortality_rate'].min()

recommendations['gap_vs_best'] = (recommendations['mortality_rate'] - best_score).round(1).astype(float).map('{:.1f}'.format)
recommendations['pct_worse_than_best'] = (
    ((recommendations['mortality_rate'] - best_score) / best_score) * 100
).round(1).astype(float).map('{:.1f}'.format)

recommendations[['Facility Name', 'city', 'mortality_rate', 
                  'patients_evaluated', 'gap_vs_best', 
                  'pct_worse_than_best', 'Compared to National',
                  'recommendation_rank']]

,Facility Name,city,mortality_rate,patients_evaluated,gap_vs_best,pct_worse_than_best,Compared to National,recommendation_rank
0,NYU LANGONE HOSPITALS,NEW YORK,6.7,633,0.0,0.0,Better Than the National Rate,1
1,MONTEFIORE MEDICAL CENTER,BRONX,9.6,275,2.9,43.3,Better Than the National Rate,2
2,NORTHERN WESTCHESTER HOSPITAL,MOUNT KISCO,9.9,178,3.2,47.8,No Different Than the National Rate,3
3,NEW YORK-PRESBYTERIAN HOSPITAL,NEW YORK,9.9,613,3.2,47.8,Better Than the National Rate,3
4,MAIMONIDES MEDICAL CENTER,BROOKLYN,10.2,399,3.5,52.2,No Different Than the National Rate,5


## Provider Recommendation: Top 5 Hospitals in NY for Heart Attack (MORT_30_AMI)

**NYU Langone is the clear #1** — not just by rank, but by magnitude. 
At 6.7% mortality, it performs **43% better than the #2 hospital** (Montefiore, 9.6%). 
The difference between #2 and #3 is only 3% — ranks look equal but the real gap is at the top.

### Why does Northern Westchester show "No Different Than National" with a 9.9% score?

The CMS comparison is not based on the raw score alone — it uses **confidence intervals**.
Northern Westchester evaluated only 178 patients, so its confidence interval is wide 
and the CMS cannot statistically confirm it outperforms the national rate.
NY-Presbyterian has the same score (9.9%) but 613 patients — enough statistical 
power to be certified "Better Than the National Rate."

This is a critical insight for provider recommendation models: **rank alone is not enough**. 
A good recommendation engine must also consider:
- Sample size (is the score reliable?)
- Magnitude of difference (is #2 meaningfully worse than #1?)
- Statistical significance (what does the CMS confidence interval say?)

In [7]:
## Case 2: Texas — 
state = "TX"
condition = "MORT_30_PN"
print(f"Finding top hospitals in {state} for {condition}")

Finding top hospitals in TX for MORT_30_PN


In [8]:
# Top 5 hospital recommendation for a given state and condition
recommendations = con.execute(f"""
    WITH eligible_hospitals AS (
        SELECT
            c."Facility ID",
            c."Facility Name",
            h."City/Town"        as city,
            h."State",
            h."Hospital Type",
            h."Hospital Ownership",
            c."Measure ID",
            c."Measure Name",
            CAST(c."Score" AS FLOAT)         as score,
            CAST(c."Denominator" AS INTEGER)  as denominator,
            c."Compared to National"
        FROM complications c
        JOIN hospitals h ON c."Facility ID" = h."Facility ID"
        WHERE h."State" = '{state}'
          AND c."Measure ID" = '{condition}'
          AND c."Score" NOT IN ('Not Available', 'Not Applicable')
          AND c."Denominator" NOT IN ('Not Available', 'Not Applicable')
          AND CAST(c."Denominator" AS INTEGER) >= 25
    )
    SELECT
        "Facility Name",
        city,
        "Hospital Type",
        "Hospital Ownership",
        "Measure Name",
        score                                           as mortality_rate,
        denominator                                     as patients_evaluated,
        "Compared to National",
        RANK() OVER (ORDER BY score ASC)                as recommendation_rank
    FROM eligible_hospitals
    ORDER BY recommendation_rank
    LIMIT 5
""").df()

recommendations

,Facility Name,city,Hospital Type,Hospital Ownership,Measure Name,mortality_rate,patients_evaluated,Compared to National,recommendation_rank
0,HOUSTON METHODIST HOSPITAL,HOUSTON,Acute Care Hospitals,Voluntary non-profit - Private,Death rate for pneumonia patients,8.9,1051,Better Than the National Rate,1
1,DALLAS VA MEDICAL CENTER (VA NORTH TEXAS HEALT...,DALLAS,Acute Care - Veterans Administration,Veterans Health Administration,Death rate for pneumonia patients,9.6,411,Better Than the National Rate,2
2,UT OF TEXAS SOUTHWESTERN UNIVERSITY HOSPITAL ...,DALLAS,Acute Care Hospitals,Government - State,Death rate for pneumonia patients,10.3,540,Better Than the National Rate,3
3,HOUSTON METHODIST SUGARLAND HOSPITAL,SUGAR LAND,Acute Care Hospitals,Voluntary non-profit - Private,Death rate for pneumonia patients,10.8,873,Better Than the National Rate,4
4,HOUSTON METHODIST THE WOODLANDS HOSPITAL,THE WOODLANDS,Acute Care Hospitals,Voluntary non-profit - Private,Death rate for pneumonia patients,11.0,626,Better Than the National Rate,5


In [9]:
# Add gap vs #1 hospital to show non-linear differences
best_score = recommendations['mortality_rate'].min()

recommendations['gap_vs_best'] = (recommendations['mortality_rate'] - best_score).round(1).astype(float).map('{:.1f}'.format)
recommendations['pct_worse_than_best'] = (
    ((recommendations['mortality_rate'] - best_score) / best_score) * 100
).round(1).astype(float).map('{:.1f}'.format)

recommendations[['Facility Name', 'city', 'mortality_rate', 
                  'patients_evaluated', 'gap_vs_best', 
                  'pct_worse_than_best', 'Compared to National',
                  'recommendation_rank']]

,Facility Name,city,mortality_rate,patients_evaluated,gap_vs_best,pct_worse_than_best,Compared to National,recommendation_rank
0,HOUSTON METHODIST HOSPITAL,HOUSTON,8.9,1051,0.0,0.0,Better Than the National Rate,1
1,DALLAS VA MEDICAL CENTER (VA NORTH TEXAS HEALT...,DALLAS,9.6,411,0.7,7.9,Better Than the National Rate,2
2,UT OF TEXAS SOUTHWESTERN UNIVERSITY HOSPITAL ...,DALLAS,10.3,540,1.4,15.7,Better Than the National Rate,3
3,HOUSTON METHODIST SUGARLAND HOSPITAL,SUGAR LAND,10.8,873,1.9,21.3,Better Than the National Rate,4
4,HOUSTON METHODIST THE WOODLANDS HOSPITAL,THE WOODLANDS,11.0,626,2.1,23.6,Better Than the National Rate,5


In [10]:
# How many hospitals are available per state for this condition?
con.execute(f"""
    SELECT 
        h."State",
        COUNT(DISTINCT c."Facility ID") as total_hospitals,
        COUNT(DISTINCT CASE WHEN c."Score" NOT IN ('Not Available', 'Not Applicable')
                            AND CAST(c."Denominator" AS INTEGER) >= 25 
                            THEN c."Facility ID" END) as eligible_hospitals
    FROM complications c
    JOIN hospitals h ON c."Facility ID" = h."Facility ID"
    WHERE c."Measure ID" = 'MORT_30_HF'
    GROUP BY h."State"
    ORDER BY eligible_hospitals DESC
""").df()

,State,total_hospitals,eligible_hospitals
0,CA,338,252
1,TX,406,208
2,FL,191,167
3,PA,161,134
4,NY,162,134
5,IL,177,128
6,OH,164,126
7,MI,131,98
8,IN,117,90
9,NC,106,89


## Case 2: Texas — Pneumonia (MORT_30_PN)

**Houston Methodist Hospital leads** at 8.9% mortality with 1,051 patients — 
a large, high-volume center with strong statistical confidence.

Unlike the NY case, **all 5 recommended hospitals are certified "Better Than 
the National Rate"** — TX has a more consistent top tier for pneumonia care.

The Houston Methodist system appears 3 times in the top 5 (Houston, Sugar Land, 
The Woodlands), suggesting strong system-wide quality standards — 
a useful signal for employer benefit design.

The gap between #1 and #5 is only 23.6% — far more compressed than NY (52%), 
meaning the top Texas hospitals are more similar in quality to each other.

In [11]:
# Context for MS
con.execute(f"""
    SELECT 
        h."State",
        COUNT(DISTINCT c."Facility ID") as total_hospitals,
        COUNT(DISTINCT CASE WHEN c."Score" NOT IN ('Not Available', 'Not Applicable')
                            AND CAST(c."Denominator" AS INTEGER) >= 25 
                            THEN c."Facility ID" END) as eligible_hospitals
    FROM complications c
    JOIN hospitals h ON c."Facility ID" = h."Facility ID"
    WHERE c."Measure ID" = 'MORT_30_HF'
      AND h."State" = 'MS'
    GROUP BY h."State"
""").df()

,State,total_hospitals,eligible_hospitals
0,MS,98,54


In [12]:
## Case 3: Mississippi — Heart Failure
state = "MS"
condition = "MORT_30_HF"
print(f"Finding top hospitals in {state} for {condition}")

Finding top hospitals in MS for MORT_30_HF


In [13]:
# Top 5 hospital recommendation for a given state and condition
recommendations = con.execute(f"""
    WITH eligible_hospitals AS (
        SELECT
            c."Facility ID",
            c."Facility Name",
            h."City/Town"        as city,
            h."State",
            h."Hospital Type",
            h."Hospital Ownership",
            c."Measure ID",
            c."Measure Name",
            CAST(c."Score" AS FLOAT)         as score,
            CAST(c."Denominator" AS INTEGER)  as denominator,
            c."Compared to National"
        FROM complications c
        JOIN hospitals h ON c."Facility ID" = h."Facility ID"
        WHERE h."State" = '{state}'
          AND c."Measure ID" = '{condition}'
          AND c."Score" NOT IN ('Not Available', 'Not Applicable')
          AND c."Denominator" NOT IN ('Not Available', 'Not Applicable')
          AND CAST(c."Denominator" AS INTEGER) >= 25
    )
    SELECT
        "Facility Name",
        city,
        "Hospital Type",
        "Hospital Ownership",
        "Measure Name",
        score                                           as mortality_rate,
        denominator                                     as patients_evaluated,
        "Compared to National",
        RANK() OVER (ORDER BY score ASC)                as recommendation_rank
    FROM eligible_hospitals
    ORDER BY recommendation_rank
    LIMIT 5
""").df()

recommendations

,Facility Name,city,Hospital Type,Hospital Ownership,Measure Name,mortality_rate,patients_evaluated,Compared to National,recommendation_rank
0,G. V. (SONNY) MONTGOMERY VA MEDICAL CENTER (JA...,JACKSON,Acute Care - Veterans Administration,Veterans Health Administration,Death rate for heart failure patients,9.3,247,No Different Than the National Rate,1
1,UNIVERSITY OF MISSISSIPPI MED CENTER,JACKSON,Acute Care Hospitals,Government - State,Death rate for heart failure patients,9.5,264,No Different Than the National Rate,2
2,UNIVERSITY OF MISSISSIPPI MEDICAL CENTER- GRENADA,GRENADA,Acute Care Hospitals,Government - State,Death rate for heart failure patients,10.0,74,No Different Than the National Rate,3
3,MERIT HEALTH RIVER REGION,VICKSBURG,Acute Care Hospitals,Proprietary,Death rate for heart failure patients,10.3,208,No Different Than the National Rate,4
4,HIGHLAND COMMUNITY HOSPITAL,PICAYUNE,Acute Care Hospitals,Government - Local,Death rate for heart failure patients,10.6,41,No Different Than the National Rate,5


## Case 3: Mississippi — Heart Failure (MORT_30_HF)

Mississippi is the worst-performing state overall, yet its **top 5 hospitals 
show surprisingly competitive mortality rates** (9.3–10.6%) — comparable to 
top performers in NY and TX.

Key observations:

**None are certified "Better Than the National Rate"** — all 5 show "No Different 
Than the National Rate." This reflects the small patient volumes typical of 
Mississippi hospitals; even good scores lack the statistical power to be 
certified as above average.

**The VA leads again** — G.V. Montgomery VA Medical Center in Jackson ranks #1 
at 9.3%, consistent with the national VA pattern we observed.

**Context matters:** Top 5 in MS represents the top 9.3% of 54 eligible hospitals, 
vs top 2.4% in TX (208 hospitals) and top 3.7% in NY (134 hospitals). 
A recommendation engine must communicate this context — being #1 in MS 
is a very different signal than being #1 in NY.